# VGGT — Visual Geometry Grounded Transformer (Colab)

Run [VGGT](https://github.com/facebookresearch/vggt) in Google Colab to turn a handful of images (or a short video) into a **3D point cloud + camera poses**, exported as a `.glb` file you can open in any 3D viewer.

**Before you start:** switch to a GPU runtime (`Runtime → Change runtime type → GPU`, prefer T4/L4/A100). VGGT needs CUDA.

Pipeline:
1. Install dependencies + clone repo
2. Download the pretrained `VGGT-1B` checkpoint (auto-cached after first run)
3. Provide input images (upload, video, or built-in example)
4. Run feed-forward inference (typically <1 s on H100, a few seconds on T4)
5. Convert predictions → GLB scene (point cloud + camera frustums)
6. Preview inline + download the `.glb`

## 1. Verify GPU

In [ ]:
!nvidia-smi

## 2. Clone the VGGT repo and install dependencies

Colab already ships with a recent PyTorch + CUDA, so we install VGGT's lightweight extras only (skip the pinned `torch==2.3.1` to avoid clobbering Colab's runtime).

In [ ]:
%cd /content
![ -d vggt ] || git clone https://github.com/facebookresearch/vggt.git
%cd /content/vggt

In [ ]:
# Core deps (skip torch/torchvision pin — keep Colab's CUDA-matched build)
!pip install -q numpy Pillow huggingface_hub einops safetensors
# Visualization deps for GLB export and point-cloud rendering
!pip install -q trimesh opencv-python scipy matplotlib requests
# (Optional) sky filtering uses onnxruntime
!pip install -q onnxruntime

In [ ]:
# Make the repo importable
import sys, os
sys.path.insert(0, '/content/vggt')
os.chdir('/content/vggt')
print('Working dir:', os.getcwd())

## 3. Load the VGGT-1B model

First call downloads ~5 GB of weights from Hugging Face (cached afterwards). Use the `-Commercial` variant if you need commercial-use rights — see the README.

In [ ]:
import torch
from vggt.models.vggt import VGGT

device = 'cuda' if torch.cuda.is_available() else 'cpu'
assert device == 'cuda', 'GPU runtime required — Runtime → Change runtime type → GPU'

# bfloat16 on Ampere+ (T4 is sm_75 → falls back to float16)
dtype = torch.bfloat16 if torch.cuda.get_device_capability()[0] >= 8 else torch.float16
print(f'Device: {device} | dtype: {dtype}')

print('Loading VGGT-1B (downloads on first run, ~5 GB)...')
model = VGGT.from_pretrained('facebook/VGGT-1B').to(device)
model.eval()
print('Model ready.')

## 4. Provide input images

Pick **one** of the three options below and run that cell. All of them populate the folder `/content/scene/images/`.

### Option A — Upload your own images

In [ ]:
import os, shutil
from google.colab import files

scene_dir = '/content/scene'
img_dir = os.path.join(scene_dir, 'images')
if os.path.exists(scene_dir):
    shutil.rmtree(scene_dir)
os.makedirs(img_dir, exist_ok=True)

uploaded = files.upload()  # opens the file picker
for name, data in uploaded.items():
    with open(os.path.join(img_dir, name), 'wb') as f:
        f.write(data)
print(f'Saved {len(uploaded)} images to {img_dir}')

### Option B — Upload a short video (1 frame/sec is extracted)

In [ ]:
import os, shutil, cv2
from google.colab import files

scene_dir = '/content/scene'
img_dir = os.path.join(scene_dir, 'images')
if os.path.exists(scene_dir):
    shutil.rmtree(scene_dir)
os.makedirs(img_dir, exist_ok=True)

uploaded = files.upload()
video_path = next(iter(uploaded))  # take the first uploaded file
with open(video_path, 'wb') as f:
    f.write(uploaded[video_path])

FRAMES_PER_SECOND = 1   # bump this for denser sampling
vs = cv2.VideoCapture(video_path)
fps = vs.get(cv2.CAP_PROP_FPS) or 30
stride = max(1, int(fps / FRAMES_PER_SECOND))

saved, count = 0, 0
while True:
    ok, frame = vs.read()
    if not ok:
        break
    if count % stride == 0:
        cv2.imwrite(os.path.join(img_dir, f'{saved:06d}.png'), frame)
        saved += 1
    count += 1
vs.release()
print(f'Extracted {saved} frames into {img_dir} (input fps={fps:.1f})')

### Option C — Use a built-in example (kitchen scene shipped with the repo)

In [ ]:
import os, shutil, glob

scene_dir = '/content/scene'
img_dir = os.path.join(scene_dir, 'images')
if os.path.exists(scene_dir):
    shutil.rmtree(scene_dir)
os.makedirs(img_dir, exist_ok=True)

src_dirs = [
    '/content/vggt/examples/kitchen/images',
    '/content/vggt/examples/room/images',
]
src = next((d for d in src_dirs if os.path.isdir(d) and glob.glob(os.path.join(d, '*'))), None)
if src is None:
    raise FileNotFoundError('No example images shipped with this checkout — use Option A or B.')

for p in sorted(glob.glob(os.path.join(src, '*'))):
    shutil.copy(p, img_dir)
print(f'Copied {len(os.listdir(img_dir))} images from {src}')

## 5. Run VGGT inference

This is the actual 3D reconstruction. Output keys (after squeezing the batch dim):
- `pose_enc`, `extrinsic`, `intrinsic` — camera parameters
- `depth` `(S,H,W,1)`, `depth_conf` `(S,H,W)`
- `world_points` `(S,H,W,3)`, `world_points_conf` `(S,H,W)` — direct point-map prediction
- `world_points_from_depth` `(S,H,W,3)` — depth unprojected through cameras (often crisper)
- `images` — preprocessed input tensor (kept for visualization)

In [ ]:
import glob, time, gc
import numpy as np
import torch
from vggt.utils.load_fn import load_and_preprocess_images
from vggt.utils.pose_enc import pose_encoding_to_extri_intri
from vggt.utils.geometry import unproject_depth_map_to_point_map

image_names = sorted(glob.glob(os.path.join(img_dir, '*')))
assert len(image_names) > 0, f'No images found in {img_dir}'
print(f'Loading {len(image_names)} images...')

images = load_and_preprocess_images(image_names).to(device)
print(f'Image batch shape: {tuple(images.shape)}  (S, 3, H, W)')

t0 = time.time()
with torch.no_grad():
    with torch.cuda.amp.autocast(dtype=dtype):
        predictions = model(images)
torch.cuda.synchronize()
print(f'VGGT forward pass: {time.time() - t0:.2f}s')

# Decode pose encoding into 3x4 extrinsics + 3x3 intrinsics (OpenCV convention)
extrinsic, intrinsic = pose_encoding_to_extri_intri(predictions['pose_enc'], images.shape[-2:])
predictions['extrinsic'] = extrinsic
predictions['intrinsic'] = intrinsic

# Move everything to NumPy and drop the batch dim
for k in list(predictions.keys()):
    if isinstance(predictions[k], torch.Tensor):
        predictions[k] = predictions[k].cpu().numpy().squeeze(0)
predictions['pose_enc_list'] = None

# Unproject depth -> world points (usually higher quality than the raw point head)
predictions['world_points_from_depth'] = unproject_depth_map_to_point_map(
    predictions['depth'], predictions['extrinsic'], predictions['intrinsic']
)

# Persist for later (and so you can re-render with different filters without re-running the model)
np.savez(os.path.join(scene_dir, 'predictions.npz'),
         **{k: v for k, v in predictions.items() if v is not None})

del images
gc.collect(); torch.cuda.empty_cache()
print('Predictions ready. Keys:', [k for k, v in predictions.items() if v is not None])

## 6. Build the GLB 3D scene

Tunable knobs:
- `CONF_THRES` — drop the lowest *X%* of low-confidence points (0 = keep all, 50 = drop bottom half).
- `PREDICTION_MODE` — `'Depthmap and Camera Branch'` (recommended) or `'Pointmap Branch'`.
- `SHOW_CAM` — include camera frustums.
- `MASK_BLACK_BG` / `MASK_WHITE_BG` — drop near-black / near-white pixels (handy for object-on-background shots).

In [ ]:
from visual_util import predictions_to_glb

CONF_THRES = 50.0
PREDICTION_MODE = 'Depthmap and Camera Branch'  # or 'Pointmap Branch'
SHOW_CAM = True
MASK_BLACK_BG = False
MASK_WHITE_BG = False
MASK_SKY = False  # set True for outdoor scenes (downloads skyseg.onnx on first use)

glb_path = os.path.join(scene_dir, 'scene.glb')
scene = predictions_to_glb(
    predictions,
    conf_thres=CONF_THRES,
    filter_by_frames='All',
    mask_black_bg=MASK_BLACK_BG,
    mask_white_bg=MASK_WHITE_BG,
    show_cam=SHOW_CAM,
    mask_sky=MASK_SKY,
    target_dir=scene_dir,
    prediction_mode=PREDICTION_MODE,
)
scene.export(file_obj=glb_path)
print(f'Exported: {glb_path}  ({os.path.getsize(glb_path)/1e6:.2f} MB)')

## 7. Quick inline preview (matplotlib)

Cheap orthographic preview of the point cloud, just to confirm the reconstruction looks reasonable. For real interactive viewing, download the GLB (next cell) and open it in https://gltf-viewer.donmccurdy.com or any 3D tool.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

pts = predictions['world_points_from_depth'].reshape(-1, 3)
imgs = predictions['images']  # (S, 3, H, W)
cols = np.transpose(imgs, (0, 2, 3, 1)).reshape(-1, 3)
conf = predictions['depth_conf'].reshape(-1)

thr = np.percentile(conf, CONF_THRES) if CONF_THRES > 0 else 0.0
mask = (conf >= thr) & (conf > 1e-5)
pts_v, cols_v = pts[mask], np.clip(cols[mask], 0, 1)

# Subsample for plotting speed
if len(pts_v) > 60000:
    sel = np.random.choice(len(pts_v), 60000, replace=False)
    pts_v, cols_v = pts_v[sel], cols_v[sel]

fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')
ax.scatter(pts_v[:, 0], pts_v[:, 1], pts_v[:, 2], c=cols_v, s=0.5, marker='.')
ax.set_title(f'VGGT point cloud — {len(pts_v):,} points (after conf filtering)')
ax.set_xlabel('X'); ax.set_ylabel('Y'); ax.set_zlabel('Z')
ax.view_init(elev=-60, azim=-90)  # roughly looking through the first camera
plt.tight_layout()
plt.show()

## 8. Download the GLB

In [ ]:
from google.colab import files
files.download(glb_path)

## (Optional) Re-render with different filters without re-running the model

Tweak the constants in section 6 and re-run that cell — the cached `predictions` dict is reused, so each render is just a few seconds.

## (Optional) Going further
- **Interactive viser viewer:** `!pip install viser==0.2.23 tqdm && python demo_viser.py --image_folder /content/scene/images` — but Colab can't expose the port directly; use ngrok or run locally.
- **COLMAP export for Gaussian Splatting:** `!pip install pycolmap==3.10.0 pyceres==2.3 && python demo_colmap.py --scene_dir=/content/scene` (with optional `--use_ba` for bundle adjustment).
- **Memory tip:** the aggregator is roughly linear in #frames in time and quadratic-ish in memory — 50 frames ≈ 11 GB, 100 ≈ 21 GB. Cut frame count or use a bigger GPU if you OOM.